# Streaming: SSE tokens & agent events, then two-way WebSockets

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 25 §25.3–25.4 · type: walkthrough*  🔧 *This builds streaming endpoints.*

**The promise:** you will stream an agent's output to the client — first **one-way** via Server-Sent Events (structured progress events *and* answer tokens, terminated by `[DONE]`), handle a generator that fails mid-stream, then go **two-way** with a WebSocket so the client can push a `stop` message and steer a running agent. All in-process, no port, no API key.

## 🧠 Why this matters

LLM output arrives token by token, and the user should see it that way. Waiting ten seconds for a complete answer feels broken; watching it stream feels instant — **latency you can see progressing is far more tolerable than latency you can't.** That single UX move is most of what makes an agent feel trustworthy and alive instead of frozen behind a spinner.

There are two mechanisms, and choosing between them is the lesson. **SSE** is a one-way stream from server to client over plain HTTP — dead simple, perfect for relaying tokens and an agent's progress. **WebSockets** add a persistent *two-way* channel, which you need only when the client must also push to the server mid-stream — to interrupt, steer, or collaborate. Reach for the simpler tool by default; pay for WebSockets' extra complexity only when you actually need the upstream channel.

## Objectives & prerequisites

**By the end you can:**

- Return a `StreamingResponse` with `media_type="text/event-stream"` from an async generator that yields SSE frames (`data: …\n\n`), ending with `[DONE]`.
- Stream **structured progress events** ("calling tool X", "step 3 of 8") before the answer tokens — exactly what the Ch 38 frontend renders.
- Handle a generator that raises mid-stream: emit an error frame and close cleanly instead of dribbling a half-answer.
- Run a `@app.websocket` handler that receives a message, streams events back, and honors a client `stop` pushed mid-run.

**Prereqs:** `25-01-fastapi-from-zero.ipynb`. Packages: `fastapi`, `httpx`, `httpx-sse` is *not* required (we parse frames by hand to see their shape) — only `requirements.txt` packages plus stdlib.

**Run first:** the Setup cell. Defaults to `MOCK=1`; the token/event stream is canned from `data/stream-events.json`, so the notebook is **free, offline, and deterministic**. The WebSocket is exercised via the ASGI test client — no real socket server.

In [ ]:
# --- Setup -------------------------------------------------------------------
import asyncio
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # secrets (if any) come from the environment only

# MOCK=1 (default): the token/event stream is canned -> free, offline,
# deterministic. MOCK=0 would stream one short live completion (see stream_agent).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(25)  # determinism for anything stochastic

MODEL = "claude-opus-4-8"  # the book's default model (only used if MOCK=0)
DATA_DIR = Path("data")    # tiny committed fixtures live here

# Load the scripted stream once so every cell is deterministic.
SCRIPT = json.loads((DATA_DIR / "stream-events.json").read_text(encoding="utf-8"))
EVENTS = SCRIPT["events"]   # structured progress events
TOKENS = SCRIPT["tokens"]   # answer tokens

if MOCK:
    print(f"MOCK mode: streaming {len(EVENTS)} canned events + {len(TOKENS)} tokens. Offline, free.")
else:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError(
            "COMPANION_MOCK=0 but ANTHROPIC_API_KEY is not set. "
            "Set it in your environment/.env, or use COMPANION_MOCK=1."
        )
    print(f"LIVE mode: stream_agent will stream one short completion from {MODEL}.")

## 1 · The agent as an async event stream

Everything downstream consumes one async generator: `stream_agent` yields **structured events first** (status, tool calls, retrieval counts), then **answer tokens**. The MOCK path replays the scripted fixture; the live path shows the real streamed call you'd make with `COMPANION_MOCK=0`. Keeping both behind one generator means the endpoints below never know or care which is running.

In [ ]:
async def stream_agent(question: str):
    """Yield agent events then answer tokens, as dicts. MOCK replays the fixture;
    live streams a real completion. Each yielded dict becomes one SSE frame."""
    if MOCK:
        for ev in EVENTS:
            await asyncio.sleep(0)  # a real yield point; instant + deterministic
            yield ev
        for tok in TOKENS:
            await asyncio.sleep(0)
            yield {"type": "token", "text": tok}
        return
    import anthropic  # imported only on the live path

    client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY
    async with client.messages.stream(
        model=MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": question}],
    ) as stream:
        async for text in stream.text_stream:
            yield {"type": "token", "text": text}


# Peek at the shape (consume the async generator in this cell).
preview = [ev async for ev in stream_agent("refund window?")]
print("first event :", preview[0])
print("first token :", preview[len(EVENTS)])
print("total frames:", len(preview))

## 2 · 🔧 `StreamingResponse` over `text/event-stream`

Here is the SSE endpoint from the book. An inner async generator wraps each event as an **SSE frame** — the wire format is literally `data: <payload>\n\n` (the blank line terminates the frame) — and we close the stream with the conventional `data: [DONE]\n\n` sentinel so the client knows to stop reading. `StreamingResponse` with `media_type="text/event-stream"` is all FastAPI needs to keep the connection open and flush each frame as it's produced.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI(title="Streaming agent API (Ch 25 §25.3)")


class AskRequest(BaseModel):
    question: str


def sse_frame(payload: dict) -> str:
    """One Server-Sent Event frame: `data: <json>` then a blank line."""
    return f"data: {json.dumps(payload)}\n\n"


@app.post("/ask/stream")
async def ask_stream(req: AskRequest):
    async def events():
        async for ev in stream_agent(req.question):
            yield sse_frame(ev)          # structured event OR token, same frame shape
        yield "data: [DONE]\n\n"        # the sentinel that closes the stream

    return StreamingResponse(events(), media_type="text/event-stream")


print("endpoint registered: POST /ask/stream -> text/event-stream")

## 3 · Consume the stream: structured events, *then* tokens

We drive it with `httpx` over the in-process ASGI app (no port). `client.stream(...)` lets us read the response **incrementally** with `iter_lines()`, parsing each `data:` line as it arrives — exactly what a browser's `EventSource` or the Ch 38 frontend does. Watch the order: progress events come first (so the UI can show the agent *working*), then the answer tokens, then `[DONE]`.

In [ ]:
import httpx


async def consume_sse(path: str, body: dict):
    """Stream the SSE endpoint and print each frame as it arrives."""
    transport = httpx.ASGITransport(app=app)  # in-process; binds no socket
    answer = ""
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as ac:
        async with ac.stream("POST", path, json=body) as resp:
            print("content-type:", resp.headers["content-type"], "\n")
            async for line in resp.aiter_lines():
                if not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    print("  <- [DONE] (stream closed)")
                    break
                ev = json.loads(data)
                if ev["type"] == "token":
                    answer += ev["text"]
                    print(f"  token : {ev['text']!r}")
                else:
                    print(f"  event : {ev}")
    return answer


final = await consume_sse("/ask/stream", {"question": "What is the refund window?"})
print("\nassembled answer:", repr(final))

**What you just saw.** The client received the structured progress events *before* a single answer token — that ordering is the whole point. A frontend can render "planning… retrieved 5 sources… drafting…" so the user watches the agent think and act, then sees the answer assemble token by token. You streamed both **over the same SSE channel**, with the same frame shape, ending on `[DONE]`.

## 4 · 🔮 Predict: the generator raises mid-stream

Streaming has a hazard buffered responses don't: by the time something fails, you've **already sent part of the output**. You can't take it back with a clean `500` — the status line went out long ago. So what should the client see if the agent blows up after three tokens?

**Predict before running:** if our generator raises halfway through, and we *don't* handle it, what does the client get — a clear error, or a silently truncated answer that looks complete? Then run the handled version and compare.

In [ ]:
async def flaky_agent(question: str, fail_after: int = 3):
    """Yields a few tokens, then raises — simulating a mid-stream agent failure."""
    for i, tok in enumerate(TOKENS):
        if i == fail_after:
            raise RuntimeError("upstream model dropped the connection")
        await asyncio.sleep(0)
        yield {"type": "token", "text": tok}


@app.post("/ask/stream-safe")
async def ask_stream_safe(req: AskRequest):
    async def events():
        try:
            async for ev in flaky_agent(req.question):
                yield sse_frame(ev)
        except Exception as exc:  # noqa: BLE001 — we deliberately convert any failure
            # Partial-output hygiene: emit a typed error frame, then close cleanly.
            yield sse_frame({"type": "error", "message": str(exc)})
        finally:
            yield "data: [DONE]\n\n"  # always terminate so the client stops reading

    return StreamingResponse(events(), media_type="text/event-stream")


await consume_sse("/ask/stream-safe", {"question": "refund?"})

**What you just saw.** The client got the first three tokens, then an explicit `{"type": "error", …}` frame, then `[DONE]`. That's **partial-output hygiene**: the UI can show "the answer was cut short" instead of presenting three tokens as if they were the whole, correct response. Without the `try/except/finally`, the connection would just die after token three — the client can't distinguish that from a complete answer, which is how you ship a bug that silently truncates results. Always wrap a streaming generator and always emit a terminator.

## 5 · 🔧 Two-way: a WebSocket with a mid-run `stop`

SSE can't hear the client mid-stream. When the user must **interrupt or steer** a running agent, you need the upstream channel a WebSocket gives you. This handler from the book accepts the socket, reads the question, then streams events back with `send_json` — but it *also* watches for a client `stop` message arriving on the same connection and cancels the run when it sees one. That's the capability SSE simply cannot provide.

In [ ]:
from fastapi import WebSocket, WebSocketDisconnect


@app.websocket("/ws/agent")
async def agent_ws(ws: WebSocket):
    await ws.accept()
    try:
        question = await ws.receive_text()              # client pushes the request
        gen = stream_agent(question)

        async def pump():
            async for ev in gen:
                await ws.send_json(ev)                   # push progress + tokens live
            await ws.send_json({"type": "done"})

        async def watch_stop():
            # Listen for an upstream 'stop' WHILE the agent streams — the SSE-impossible move.
            msg = await ws.receive_text()
            if msg == "stop":
                return "stopped"
            return msg

        pump_task = asyncio.create_task(pump())
        stop_task = asyncio.create_task(watch_stop())
        done, pending = await asyncio.wait(
            {pump_task, stop_task}, return_when=asyncio.FIRST_COMPLETED
        )
        for t in pending:
            t.cancel()  # whichever finished first wins; cancel the other
        if stop_task in done and stop_task.result() == "stopped":
            await ws.send_json({"type": "stopped", "message": "run cancelled by client"})
    except WebSocketDisconnect:
        pass  # client closed the socket; nothing to clean up in this toy


print("endpoint registered: websocket /ws/agent")

### Drive the WebSocket with the test client

FastAPI's `TestClient` exercises a WebSocket in-process via `websocket_connect` — no real socket server, no port. We connect, send a question, read a couple of events, then **push `stop`** and confirm the server cancels the run and acknowledges. This is the interactive loop a steerable agent UI runs.

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

with client.websocket_connect("/ws/agent") as ws:
    ws.send_text("What is the refund window?")
    # Read the first two streamed events, then change our mind and interrupt.
    print("recv:", ws.receive_json())
    print("recv:", ws.receive_json())
    ws.send_text("stop")            # <-- upstream message mid-run (SSE can't do this)
    # Drain until we see the server's acknowledgement of the stop.
    for _ in range(len(EVENTS) + len(TOKENS) + 2):
        msg = ws.receive_json()
        print("recv:", msg)
        if msg.get("type") in {"stopped", "done"}:
            break

**What you just saw.** The server streamed events, and the moment we pushed `stop` it cancelled the run and sent a `stopped` acknowledgement — a round trip *during* generation. That bidirectional, mid-stream control is the only reason to take on a WebSocket; if your client just reads output, SSE already does the job with far less machinery.

## ⚠️ Pitfall: reaching for WebSockets when SSE suffices

WebSockets *look* more powerful, so it's tempting to default to them. Resist it. A WebSocket is a stateful, persistent connection: you now own its full lifecycle — accept, heartbeat/ping-pong, detecting half-open connections, **reconnection** logic on the client, and back-pressure — none of which plain SSE makes you think about. SSE rides ordinary HTTP, so proxies, load balancers, and the browser's `EventSource` (with automatic reconnect) handle it for free.

The rule: **if the data flows server → client only, use SSE.** Only when the client must push *to the server mid-stream* — interrupt, steer, collaborate — do you actually need the upstream channel, and only then is the WebSocket complexity earning its keep.

## 🎯 Senior lens: choose the transport deliberately

The decision isn't about which protocol is "better"; it's about which channel your *interaction* actually needs. For the overwhelming majority of chat and agent output — tokens and progress flowing one way — **SSE is the right default**: simplest to operate, friendliest to infrastructure, trivially reconnectable. **WebSockets earn their complexity only for genuinely interactive sessions**: an interruptible agent, a live cursor in a collaborative doc, a voice loop. A useful heuristic: start with SSE; introduce a WebSocket the first time you write a feature where the client must *talk back during* the stream, and not a moment sooner. Picking WebSockets early means paying the connection-lifecycle tax on every endpoint, forever, for a channel most of them never use.

## Recap

- **SSE = one-way stream over plain HTTP.** Return a `StreamingResponse(media_type="text/event-stream")` from an async generator that yields `data: …\n\n` frames, ended by `[DONE]`.
- **Stream structured events, not just tokens** — "calling tool X", "step 3 of 8", *then* the answer — so the UI shows the agent thinking and acting (rendered in Ch 38).
- **Partial-output hygiene:** wrap the generator in `try/except/finally`, emit a typed `error` frame on failure, and always send the terminator. A dropped stream must not look like a finished answer.
- **WebSockets add a two-way channel** — `accept`, `receive`, `send_json` — for clients that push mid-run (e.g. `stop`). Exercise them with `TestClient.websocket_connect`, no real socket.
- **Choose deliberately:** SSE for server→client output (the default); WebSockets only when the client must talk back during the stream.

## Exercises

Each one *changes something* and asks you to predict first. (Solutions land in `solutions/` in Phase 2.)

1. **Heartbeat frames.** Long-lived SSE streams can be dropped by idle-timeout proxies. Add a `: keep-alive\n\n` comment frame (SSE comments start with `:`) every few events in `/ask/stream`. Predict whether your `consume_sse` parser will mistake it for data, then adjust the parser to ignore comment lines.
2. **Event `id`/`event:` fields.** SSE frames may carry `event: <name>` and `id: <n>` lines so the browser can route by type and resume after reconnect. Extend `sse_frame` to emit `event: token` vs `event: status`, send it, and predict how a browser `EventSource` would dispatch them differently.
3. **WebSocket steer, not just stop.** Change `watch_stop` to accept a `{"action": "refine", "hint": …}` message and, instead of cancelling, restart `stream_agent` with the hint folded into the question. Predict what the client sees on the wire when it steers mid-run.

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here


## Next

You can stream tokens and structured events, and steer an agent over a socket. Next: assemble these into the capstone's real API surface, with the queue boundary in the right place.

- ▶️ **Next notebook:** [`25-03-capstone-backend-api.ipynb`](./25-03-capstone-backend-api.ipynb) — the chapter's 🔧 Build: `POST /runs`, `GET /runs/{id}`, the SSE progress stream, and the thin-API/durable-worker split.
- 🔧 **Template (the production version of this):** [`../../../templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/) — SSE and WebSocket routes as a real scaffold. You built the toy; that's the real one.
- 🎓 **Capstone:** these streaming routes become `capstone/app/`; checkpoint `checkpoints/ch25-backend-api`. The frontend that renders these events is Ch 38.
- 📖 **Book:** §25.3 (streaming & SSE) and §25.4 (WebSockets).